In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1n3eBiI-Pwo16Uwm9iKj4X-IsXY8Qo8Gb", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup Check
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup_check.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Llm Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_llm_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Notebook Title
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_notebook_title.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# Few-Shot Learning & In-Context Learning

*Part 2 of the Vizuara series on Prompt Design Principles*
*Estimated time: 55 minutes*

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/context-engineering/prompt-design-principles/practice/2/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you are teaching a new employee to categorize support tickets. You could write them a 10-page manual describing every category. Or you could show them 5 examples and say: "See the pattern? Do that."

This is **few-shot learning** — and it is one of the most powerful techniques in prompt design. Instead of describing what you want, you *show* what you want by including input-output examples directly in the prompt.

In this notebook, we will:
- Understand why few-shot learning works (in-context learning)
- Build few-shot prompts for classification, extraction, and generation
- Measure how the number and order of examples affects accuracy
- Discover the limits of few-shot learning

In [ ]:
# 🔧 Setup
!pip install -q anthropic matplotlib seaborn pandas numpy

import os
import json
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from anthropic import Anthropic

client = Anthropic()

def query_llm(system_prompt, user_message, model="claude-sonnet-4-20250514"):
    """Send a message to Claude and return the response text."""
    response = client.messages.create(
        model=model,
        max_tokens=512,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text.strip()

print("✅ Setup complete!")

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

Why does showing examples work at all? The model was not retrained — we just added text to the prompt. How can a few examples change its behavior?

The answer is **in-context learning**. When you include input-output examples in the prompt, the model identifies the pattern and applies it to the new input — without any weight updates. The examples activate relevant patterns learned during training, effectively "selecting" the right behavior from everything the model already knows.

Think of it this way. The model has learned millions of patterns during training. Without examples, it does not know *which* pattern you want. Your examples are like tuning a radio dial — they tell the model which frequency to lock onto.

### 🤔 Think About This

Suppose you want a model to classify product reviews as "positive", "negative", or "neutral". You send just: "Classify this review: 'The battery is okay but nothing special.'"

The model could output "neutral", "mixed", "somewhat positive", "3 out of 5 stars", or many other formats. It does not know your labeling scheme.

Now suppose you show it three examples first. Suddenly the model knows:
- Your exact label set (positive/negative/neutral)
- Your threshold for each category
- Your output format (just the label, no explanation)

The examples removed ambiguity. That is the power of few-shot learning.

In [ ]:
#@title 🎧 Listen: The Mathematics
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_the_mathematics.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

We can formalize in-context learning. Given a new input $x$ and $k$ demonstration examples $\{(x_1, y_1), \ldots, (x_k, y_k)\}$, the model computes:

$$P(y \mid x) \approx P\big(y \mid x, (x_1, y_1), (x_2, y_2), \ldots, (x_k, y_k)\big)$$

**What does this mean computationally?** The right-hand side is the probability of the output $y$ conditioned on both the input $x$ AND the examples. The examples narrow the distribution — they make the model more confident about the correct output format and content.

Let us see this numerically. Without examples (zero-shot):
- $P(\text{"positive"}) = 0.35$, $P(\text{"negative"}) = 0.25$, $P(\text{"neutral"}) = 0.15$, $P(\text{"mixed"}) = 0.10$, $P(\text{other}) = 0.15$

With 3 examples that use exactly "positive"/"negative"/"neutral":
- $P(\text{"positive"}) = 0.08$, $P(\text{"negative"}) = 0.02$, $P(\text{"neutral"}) = 0.88$, $P(\text{"mixed"}) = 0.01$, $P(\text{other}) = 0.01$

The examples concentrated probability mass on the correct label. This is exactly what we want.

In [ ]:
#@title 🎧 Listen: Viz Probability
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_viz_probability.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 📊 Visualization: How examples sharpen the output distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ["positive", "negative", "neutral", "mixed", "other"]

# Zero-shot distribution
zero_shot = [0.35, 0.25, 0.15, 0.10, 0.15]
colors_zero = ['#FF6B6B', '#FF6B6B', '#4ECDC4', '#FF6B6B', '#FF6B6B']
axes[0].bar(labels, zero_shot, color='#FF6B6B', alpha=0.7, edgecolor='black')
axes[0].bar(labels[2], zero_shot[2], color='#4ECDC4', alpha=0.7, edgecolor='black')
axes[0].set_title("Zero-Shot: Spread Distribution", fontsize=13)
axes[0].set_ylabel("Probability", fontsize=12)
axes[0].set_ylim(0, 1.0)
axes[0].axhline(y=0.2, color='gray', linestyle='--', alpha=0.5, label='Uniform')
axes[0].legend()

# Few-shot distribution
few_shot = [0.08, 0.02, 0.88, 0.01, 0.01]
axes[1].bar(labels, few_shot, color='#FF6B6B', alpha=0.7, edgecolor='black')
axes[1].bar(labels[2], few_shot[2], color='#4ECDC4', alpha=0.9, edgecolor='black')
axes[1].set_title("3-Shot: Peaked Distribution", fontsize=13)
axes[1].set_ylabel("Probability", fontsize=12)
axes[1].set_ylim(0, 1.0)
axes[1].annotate('Examples sharpen\nthe distribution!', xy=(2, 0.88),
                 xytext=(3.5, 0.7), fontsize=11,
                 arrowprops=dict(arrowstyle='->', color='green', lw=2),
                 color='green', fontweight='bold')

plt.suptitle("Few-Shot Examples Concentrate Probability on the Correct Answer",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Prompt Builder Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_prompt_builder_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let's Build It — Component by Component

### 4.1 A Basic Few-Shot Prompt Builder

In [ ]:
#@title 🎧 Listen: Prompt Builder Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_prompt_builder_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo Ner After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_todo_ner_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def build_few_shot_prompt(task_description, examples, new_input,
                          input_label="Input", output_label="Output"):
    """
    Build a few-shot prompt from examples.

    Args:
        task_description: What the model should do
        examples: List of (input, output) tuples
        new_input: The input to classify/process
        input_label: Label for inputs (e.g., "Review", "Code")
        output_label: Label for outputs (e.g., "Sentiment", "Category")
    """
    prompt = f"{task_description}\n\n"

    for inp, out in examples:
        prompt += f"{input_label}: {inp}\n{output_label}: {out}\n\n"

    prompt += f"{input_label}: {new_input}\n{output_label}:"
    return prompt

# Test it
examples = [
    ("The battery lasts all day and the camera is stunning.", "positive"),
    ("Crashed twice in the first hour. Waste of money.", "negative"),
    ("Fast shipping, exactly as described. Very happy.", "positive"),
]

prompt = build_few_shot_prompt(
    "Classify the sentiment of each review as positive or negative.",
    examples,
    "The interface is confusing and customer support never responds.",
    input_label="Review",
    output_label="Sentiment"
)

print(prompt)
print("\n" + "=" * 50)
response = query_llm("", prompt)
print(f"Model response: {response}")

In [ ]:
#@title 🎧 Listen: Measuring Shots Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_measuring_shots_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 Measuring the Effect of Number of Shots

In [ ]:
#@title 🎧 Listen: Measuring Shots Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_measuring_shots_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Full dataset for experiments
all_examples = [
    ("The battery lasts all day and the camera is stunning.", "positive"),
    ("Crashed twice in the first hour. Complete waste of money.", "negative"),
    ("Fast shipping, exactly as described. Very happy.", "positive"),
    ("Screen cracked after one week. Terrible quality.", "negative"),
    ("Decent product for the price. Nothing special.", "neutral"),
    ("Absolutely love it! Best purchase this year.", "positive"),
    ("Customer support ignored my complaint for weeks.", "negative"),
    ("It works as advertised. Solid but unexciting.", "neutral"),
]

test_cases = [
    ("The design is sleek but the performance is disappointing.", "negative"),
    ("Works great, exactly what I needed!", "positive"),
    ("Arrived late but the product itself is fine.", "neutral"),
    ("Do NOT buy this. Broke after 2 days.", "negative"),
    ("Amazing quality for such a low price!", "positive"),
]

def run_few_shot_experiment(n_shots, num_trials=1):
    """Run classification with n examples and measure accuracy."""
    correct = 0
    total = len(test_cases)

    for test_input, expected in test_cases:
        shot_examples = all_examples[:n_shots]
        prompt = build_few_shot_prompt(
            "Classify the sentiment as positive, negative, or neutral.",
            shot_examples, test_input,
            input_label="Review", output_label="Sentiment"
        )
        response = query_llm("", prompt).lower().strip()
        if expected in response:
            correct += 1

    return correct / total

# Run experiments with 0, 1, 2, 3, 5 shots
shot_counts = [0, 1, 2, 3, 5]
accuracies = []

for n in shot_counts:
    acc = run_few_shot_experiment(n)
    accuracies.append(acc)
    print(f"{n}-shot accuracy: {acc:.0%}")

In [ ]:
#@title 🎧 Listen: Viz Shots Accuracy
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_viz_shots_accuracy.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 📊 Visualization: Accuracy vs number of shots
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(shot_counts, accuracies, color=['#FF6B6B', '#FFA07A', '#FFD700', '#90EE90', '#4ECDC4'],
              edgecolor='black', linewidth=1.2, width=0.6)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{acc:.0%}', ha='center', fontsize=13, fontweight='bold')

ax.set_xlabel('Number of Examples (Shots)', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Few-Shot Learning: Accuracy vs. Number of Examples', fontsize=14)
ax.set_xticks(shot_counts)
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Ordering Effect Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_ordering_effect_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 The Ordering Effect

Research shows that the ORDER of examples matters — sometimes by up to 30 percentage points!

In [ ]:
#@title 🎧 Listen: Ordering Effect Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_ordering_effect_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import itertools

# Take just 3 examples and test different orderings
base_examples = all_examples[:3]
orderings = list(itertools.permutations(range(3)))

# Test a tricky input
test_input = "It's okay, not great but not terrible either."
expected = "neutral"

ordering_results = []
for perm in orderings[:6]:  # Test 6 permutations to save API calls
    ordered = [base_examples[i] for i in perm]
    prompt = build_few_shot_prompt(
        "Classify the sentiment as positive, negative, or neutral.",
        ordered, test_input,
        input_label="Review", output_label="Sentiment"
    )
    response = query_llm("", prompt).lower().strip()
    correct = expected in response
    ordering_results.append({
        "order": str(perm),
        "last_example_label": ordered[-1][1],
        "response": response[:30],
        "correct": correct
    })

df_order = pd.DataFrame(ordering_results)
print(df_order.to_string(index=False))
print(f"\nAccuracy across orderings: {df_order['correct'].mean():.0%}")
print("Notice: the last example often biases the model's output!")

In [ ]:
#@title 🎧 Listen: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_todo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. 🔧 Your Turn


In [ ]:
#@title 🎧 Listen: Todo Ner
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_todo_ner.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 1: Build a Few-Shot Named Entity Extractor

In [ ]:
def build_ner_prompt(text):
    """
    Build a few-shot prompt for Named Entity Recognition.

    Given a sentence, extract all named entities and their types.
    Use these examples to teach the model your format:

    Example 1: "Tim Cook announced the new iPhone at Apple Park in Cupertino."
    → {"persons": ["Tim Cook"], "organizations": ["Apple"], "locations": ["Apple Park", "Cupertino"], "products": ["iPhone"]}

    Example 2: "Tesla CEO Elon Musk visited the Berlin factory on Tuesday."
    → {"persons": ["Elon Musk"], "organizations": ["Tesla"], "locations": ["Berlin"], "products": []}
    """
    # ============ TODO ============
    # Build a few-shot prompt with:
    # 1. A clear task description
    # 2. At least 3 examples (the 2 above + 1 more you create)
    # 3. The new input text
    # 4. JSON output format
    # ==============================

    prompt = "???"  # YOUR CODE HERE


In [ ]:
    return prompt

# Test your NER extractor
test_texts = [
    "Sundar Pichai presented Google's Gemini model at the I/O conference in Mountain View.",
    "Microsoft acquired Activision Blizzard for $69 billion.",
]

for text in test_texts:
    prompt = build_ner_prompt(text)
    response = query_llm("", prompt)
    print(f"Text: {text}")
    print(f"Entities: {response}\n")

In [ ]:
#@title 🎧 Listen: Todo Optimal Shots
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_todo_optimal_shots.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Find the Optimal Number of Shots

In [ ]:
def find_optimal_shots(examples, test_set, max_shots=7):
    """
    Systematically test different numbers of shots to find the optimal count.

    Returns a dict mapping n_shots → accuracy.
    """
    # ============ TODO ============
    # For each value of n_shots from 0 to max_shots:
    # 1. Build a few-shot prompt with the first n examples
    # 2. Test on all items in test_set
    # 3. Record accuracy
    #
    # Return a dict: {0: 0.4, 1: 0.6, 2: 0.8, ...}
    # ==============================

    results = {}  # YOUR CODE HERE


In [ ]:
#@title 🎧 Listen: Todo Optimal Shots After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_todo_optimal_shots_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
    return results

# Test with our sentiment dataset
# results = find_optimal_shots(all_examples, test_cases)
# print("Optimal shots:", max(results, key=results.get))

In [ ]:
#@title 🎧 Listen: Putting It Together
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_putting_it_together.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together

In [ ]:
#@title 🎧 Listen: Prompt Factory Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_prompt_factory_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Complete few-shot prompt factory with best practices built in
class FewShotPromptFactory:
    """A reusable factory for building optimized few-shot prompts."""

    def __init__(self, task_description, examples, input_label="Input",
                 output_label="Output"):
        self.task_description = task_description
        self.examples = examples
        self.input_label = input_label
        self.output_label = output_label

    def build(self, new_input, n_shots=3, strategy="diverse"):
        """Build a prompt with selected examples."""
        if strategy == "first_n":
            selected = self.examples[:n_shots]
        elif strategy == "diverse":
            # Pick diverse examples (spread across labels)
            labels = list(set(ex[1] for ex in self.examples))
            selected = []
            for label in labels:
                matching = [ex for ex in self.examples if ex[1] == label]
                if matching:
                    selected.append(matching[0])
            selected = selected[:n_shots]
        else:
            selected = self.examples[:n_shots]

        return build_few_shot_prompt(
            self.task_description, selected, new_input,
            self.input_label, self.output_label
        )

# Usage
factory = FewShotPromptFactory(
    task_description="Classify the sentiment as positive, negative, or neutral.",
    examples=all_examples,
    input_label="Review",
    output_label="Sentiment"
)

prompt = factory.build("The camera is great but battery life is poor.", n_shots=3, strategy="diverse")
print(prompt)
print("\n" + "=" * 50)
print("Response:", query_llm("", prompt))

In [ ]:
#@title 🎧 Listen: Training Results Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_training_results_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Training and Results

In [ ]:
#@title 🎧 Listen: Compare Strategies Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_compare_strategies_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Compare strategies: first_n vs diverse example selection
strategies = ["first_n", "diverse"]
shot_range = [1, 2, 3, 5]

results_by_strategy = {}
for strategy in strategies:
    results_by_strategy[strategy] = []
    for n in shot_range:
        correct = 0
        for test_input, expected in test_cases:
            prompt = factory.build(test_input, n_shots=n, strategy=strategy)
            response = query_llm("", prompt).lower().strip()
            if expected in response:
                correct += 1
        acc = correct / len(test_cases)
        results_by_strategy[strategy].append(acc)
        print(f"{strategy:>10} | {n}-shot: {acc:.0%}")

# 📊 Visualization
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(shot_range))
width = 0.35

ax.bar(x - width/2, results_by_strategy["first_n"], width,
       label='First N', color='#2196F3', edgecolor='black')
ax.bar(x + width/2, results_by_strategy["diverse"], width,
       label='Diverse Selection', color='#4CAF50', edgecolor='black')

ax.set_xlabel('Number of Shots', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Example Selection Strategy Matters', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(shot_range)
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Final Output Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_final_output_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. 🎯 Final Output

In [ ]:
#@title 🎧 Listen: Final Classifier
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_final_classifier.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Build and showcase a production-grade few-shot classifier
print("=" * 60)
print("   PRODUCTION FEW-SHOT SENTIMENT CLASSIFIER")
print("=" * 60)

new_reviews = [
    "Absolutely phenomenal — exceeded every expectation!",
    "Stopped working after a month. Very frustrated.",
    "It's decent for the price. Gets the job done.",
    "The worst customer experience I've ever had.",
    "Solid build quality, fast delivery, no complaints!",
]

for review in new_reviews:
    prompt = factory.build(review, n_shots=3, strategy="diverse")
    prediction = query_llm("", prompt).strip()
    emoji = {"positive": "😊", "negative": "😞", "neutral": "😐"}.get(prediction.lower(), "❓")
    print(f"\n{emoji} [{prediction:>8}]  \"{review}\"")

print("\n" + "=" * 60)
print("🎉 Congratulations! You've built a few-shot classifier from scratch!")
print("No fine-tuning, no training data pipeline — just smart prompt design.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### 🤔 Reflection Questions
1. Why does the order of few-shot examples affect accuracy? What does this tell us about how LLMs process context?
2. When would few-shot learning fail completely? (Hint: think about tasks the model has never seen during training.)
3. How would you choose examples for a task where you have 1000 possible demonstrations but can only fit 5 in the prompt?

### 🏆 Optional Challenges
1. **Dynamic Example Selection**: Build a system that picks the most relevant examples for each new input using embedding similarity.
2. **Label Balancing**: Test whether having equal numbers of examples per label improves accuracy over random selection.
3. **Multi-Task Few-Shot**: Create a single prompt that can handle sentiment classification, entity extraction, and summarization by switching the examples.